# Cat and Dog Image Classifier

This notebook completes the freeCodeCamp Machine Learning with Python certification project. It uses TensorFlow/Keras, image augmentation, and MobileNetV2 transfer learning to classify cats and dogs.

In [ ]:
# 1 - Imports and reproducibility
import math
import os
import subprocess
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Rescaling
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
# 2 - Download the freeCodeCamp dataset
DATA_URL = "https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip"
ARCHIVE_PATH = "cats_and_dogs.zip"
PATH = "cats_and_dogs"

if not os.path.isdir(PATH):
    subprocess.run(
        ["wget", "-q", "-O", ARCHIVE_PATH, DATA_URL],
        check=True,
    )
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        archive.extractall(".")

train_dir = os.path.join(PATH, "train")
validation_dir = os.path.join(PATH, "validation")
test_dir = os.path.join(PATH, "test")

total_train = sum(len(files) for _, _, files in os.walk(train_dir))
total_val = sum(len(files) for _, _, files in os.walk(validation_dir))
total_test = len(os.listdir(test_dir))

batch_size = 64
epochs = 8
IMG_HEIGHT = 150
IMG_WIDTH = 150

print("Training images:", total_train)
print("Validation images:", total_val)
print("Test images:", total_test)

In [ ]:
# 3 - Create normalized image generators
train_image_generator = ImageDataGenerator(rescale=1.0 / 255)
validation_image_generator = ImageDataGenerator(rescale=1.0 / 255)
test_image_generator = ImageDataGenerator(rescale=1.0 / 255)

train_data_gen = train_image_generator.flow_from_directory(
    directory=train_dir,
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    shuffle=True,
    seed=42,
)

val_data_gen = validation_image_generator.flow_from_directory(
    directory=validation_dir,
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    shuffle=False,
)

# The test images live directly inside cats_and_dogs/test, so PATH is the
# generator root and "test" is supplied as its only class folder.
test_data_gen = test_image_generator.flow_from_directory(
    directory=PATH,
    classes=["test"],
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    shuffle=False,
)

In [ ]:
# 4 - Image plotting helper
def plotImages(images_arr, probabilities=False):
    fig, axes = plt.subplots(
        len(images_arr),
        1,
        figsize=(5, len(images_arr) * 3),
    )

    if len(images_arr) == 1:
        axes = [axes]

    if probabilities is False:
        for image, axis in zip(images_arr, axes):
            axis.imshow(image)
            axis.axis("off")
    else:
        for image, probability, axis in zip(
            images_arr, probabilities, axes
        ):
            axis.imshow(image)
            axis.axis("off")

            if probability > 0.5:
                axis.set_title(f"{probability * 100:.2f}% dog")
            else:
                axis.set_title(f"{(1 - probability) * 100:.2f}% cat")

    plt.tight_layout()
    plt.show()


sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])

In [ ]:
# 5 - Add data augmentation to reduce overfitting
train_image_generator = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    fill_mode="nearest",
)

In [ ]:
# 6 - Recreate the training generator with augmentation
train_data_gen = train_image_generator.flow_from_directory(
    directory=train_dir,
    batch_size=batch_size,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    shuffle=True,
    seed=42,
)

augmented_images = [
    train_data_gen[0][0][0]
    for _ in range(5)
]

plotImages(augmented_images)

In [ ]:
# 7 - Build and compile the convolutional neural network
base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

model = Sequential([
    Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    # MobileNetV2 expects values in [-1, 1]. Generators produce [0, 1].
    Rescaling(scale=2.0, offset=-1.0),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.25),
    Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# 8 - Train the model
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    x=train_data_gen,
    steps_per_epoch=math.ceil(total_train / batch_size),
    epochs=epochs,
    validation_data=val_data_gen,
    validation_steps=math.ceil(total_val / batch_size),
    callbacks=[early_stopping],
)

In [ ]:
# 9 - Plot training history
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.tight_layout()
plt.show()

In [ ]:
# 10 - Predict the 50 unlabeled test images
test_data_gen.reset()

probabilities = model.predict(
    test_data_gen,
    steps=math.ceil(total_test / batch_size),
    verbose=1,
).reshape(-1).tolist()

test_data_gen.reset()
test_images = next(test_data_gen)

plotImages(
    test_images[:len(probabilities)],
    probabilities,
)

In [ ]:
# 11 - freeCodeCamp project test
answers = [
    1, 0, 0, 1, 0, 0, 0, 0, 1, 1,
    0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
    0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
    0, 0, 1, 1, 0, 1, 1, 1, 1, 0,
    1, 0, 1, 1, 0, 0, 0, 0, 0, 0,
]

correct = 0

for probability, answer in zip(probabilities, answers):
    if round(probability) == answer:
        correct += 1

percentage_identified = (correct / len(answers)) * 100
passed_challenge = percentage_identified >= 63

print(
    "Your model correctly identified "
    f"{percentage_identified:.2f}% of the images of cats and dogs."
)

if passed_challenge:
    print("You passed the challenge!")
else:
    print(
        "You haven't passed yet. Your model should identify at least "
        "63% of the images. Keep trying. You will get it!"
    )